# Validation experiments: baseline vs regularized
Attach only processed20_v4. Enable GPU and Internet. 12 runs: two configurations x two feature sets x three seeds. Test is not loaded or evaluated. The experiment runner is embedded below; shared model/feature code uses the verified pinned bootstrap.


In [ ]:
import sys, os, json, subprocess, urllib.request, zipfile
from pathlib import Path
INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
REV = '04aa1f5'
archive = WORK/'code20.zip'
urllib.request.urlretrieve(f'https://github.com/toafn11/driver-monitoring-system/archive/{REV}.zip', archive)
destination = WORK/f'code_{REV}'
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (destination/name).resolve().is_relative_to(destination.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)
CODE = next(destination.rglob('prepare20.py')).parents[1]
os.chdir(CODE)
sys.path.insert(0, str(CODE))
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements-kaggle20.txt')],check=True)
# BEGIN VERIFIED PROJECT ACTIVATION
import importlib, hashlib
EXPECTED_CODE_HASHES = {'training/preflight20.py': '20c0fde7e542fd6cd9aa7623d6254dd57b8b45d770c0d53a3384055e2eb8c8e6', 'training/prepare20.py': '3548b813e043d57c6e06850de96c23d57e56c7f26fec346c3444546b856a6343', 'training/video_timestamps20.py': '84d5f81cc1f2bd16312ba573af251fe4ec1899e37a26882e7c80bc97168de32e', 'training/recover_cache20.py': '94cf8c6e95031df2525f1baa2a7d9422ed6b9a5239a575f7ee94acbca416eade'}
for relative, expected in EXPECTED_CODE_HASHES.items():
    actual = hashlib.sha256((CODE/relative).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code integrity mismatch: {relative}; re-import the notebook')
# Replacing sys.path alone does not replace modules retained by a notebook kernel.
for name in list(sys.modules):
    if name in ('training','core','models') or name.startswith(('training.','core.','models.')):
        del sys.modules[name]
sys.path.insert(0, str(CODE))
importlib.invalidate_caches()
for name in ('training.preflight20','training.prepare20','training.video_timestamps20'):
    module = importlib.import_module(name)
    actual_path = Path(module.__file__).resolve()
    expected_path = (CODE/Path(*name.split('.'))).with_suffix('.py').resolve()
    if actual_path != expected_path:
        raise RuntimeError(f'Wrong module loaded: {name}: {actual_path}')
    print('Verified module:', name, actual_path)
print('Verified timestamp version:', sys.modules['training.video_timestamps20'].TIMESTAMP_VERSION)
# END VERIFIED PROJECT ACTIVATION
print('Code revision VERIFIED:', REV)


In [ ]:
"""Validation-only baseline/regularization comparison; never loads test.npz."""
import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

from models.temporal20 import Temporal20, save_checkpoint, load_checkpoint
from training.train20 import fit_normalizer, metrics, predict_batches


def run_experiments(data, output, seeds=(42, 43, 44), epochs=40, patience=8):
    data, output = Path(data), Path(output)
    if output.exists():
        raise ValueError('Use a new experiment directory')
    if not torch.cuda.is_available():
        raise RuntimeError('Enable GPU before running these experiments')
    metadata = json.loads((data / 'metadata.json').read_text())
    splits = {}
    for split in ('train', 'val'):
        with np.load(data / f'{split}.npz', allow_pickle=False) as a:
            splits[split] = {k: a[k] for k in ('X', 'y', 'subjects', 'videos')}
        d = splits[split]
        assert d['X'].shape == (len(d['y']), 40, 20)
        assert np.isfinite(d['X']).all()
        assert set(d['y'].tolist()) == {0, 1}
    tr, va = splits['train'], splits['val']
    for key in ('subjects', 'videos'):
        assert not set(tr[key]) & set(va[key]), f'Overlapping {key}'
    fingerprint = hashlib.sha256((data / 'metadata.json').read_bytes()).hexdigest()
    configs = {
        'baseline': dict(hidden=64, dropout=0.2, lr=1e-3, weight_decay=1e-3),
        'regularized': dict(hidden=32, dropout=0.4, lr=3e-4, weight_decay=1e-2),
    }
    output.mkdir(parents=True)
    results = []
    for name, config in configs.items():
        for dim in (12, 20):
            for seed in seeds:
                random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
                torch.backends.cudnn.benchmark = False
                torch.backends.cudnn.deterministic = True
                folder = output / f'{name}_gru{dim}_seed{seed}'
                folder.mkdir()
                model = Temporal20(metadata['classes'], hidden=config['hidden'],
                                   dropout=config['dropout'], feature_dim=dim)
                fit_normalizer(model, tr['X']); model.to('cuda')
                counts = np.bincount(tr['y'])
                sampler = WeightedRandomSampler(
                    torch.as_tensor(1 / counts[tr['y']], dtype=torch.double),
                    len(tr['y']), replacement=True,
                    generator=torch.Generator().manual_seed(seed))
                loader = DataLoader(TensorDataset(torch.from_numpy(tr['X']),
                    torch.from_numpy(tr['y'])), batch_size=128, sampler=sampler,
                    num_workers=0, pin_memory=True)
                optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'],
                                              weight_decay=config['weight_decay'])
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='max', patience=3, factor=0.5)
                loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
                history, best, stale = [], -1., 0
                (folder / 'run.json').write_text(json.dumps(dict(
                    config=config, seed=seed, feature_dim=dim, epochs=epochs,
                    patience=patience, data_metadata_sha256=fingerprint), indent=2))
                for epoch in range(1, epochs + 1):
                    model.train(); total = 0.
                    for x, y in loader:
                        x, y = x.cuda(), y.cuda()
                        optimizer.zero_grad(set_to_none=True)
                        logits, _ = model(x); loss = loss_fn(logits, y)
                        if not torch.isfinite(loss): raise ValueError('Nonfinite loss')
                        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                        optimizer.step(); total += loss.item() * len(y)
                    probs = predict_batches(model, va['X'], 'cuda', 128)
                    report = metrics(va['y'], probs.argmax(1), metadata['classes'])
                    score = report['macro_f1']; scheduler.step(score)
                    history.append(dict(epoch=epoch, train_loss=total/len(tr['y']), validation=report))
                    (folder / 'history.json').write_text(json.dumps(history, indent=2))
                    print(folder.name, epoch, round(score, 4), flush=True)
                    if score > best + 1e-6:
                        best, stale = score, 0
                        save_checkpoint(folder / 'best.pt', model, metadata['preprocessing'],
                            epoch=epoch, seed=seed, validation=report, data_metadata_sha256=fingerprint)
                    else:
                        stale += 1
                        if stale >= patience: break
                model, saved = load_checkpoint(folder / 'best.pt', 'cuda')
                probs = predict_batches(model, va['X'], 'cuda', 128)
                pred = probs.argmax(1)
                report = metrics(va['y'], pred, metadata['classes'])
                report['by_subject'] = {str(s): metrics(va['y'][va['subjects']==s],
                    pred[va['subjects']==s], metadata['classes']) for s in sorted(set(va['subjects']))}
                (folder / 'validation.json').write_text(json.dumps(report, indent=2))
                np.savez_compressed(folder / 'validation_predictions.npz', probabilities=probs,
                                    y=va['y'], subjects=va['subjects'], videos=va['videos'])
                results.append(dict(config=name, feature_dim=dim, seed=seed,
                    best_epoch=saved['epoch'], macro_f1=report['macro_f1'],
                    drowsy_recall=report['per_class']['DROWSY']['recall']))
                (output / 'results.json').write_text(json.dumps(results, indent=2))
    summary = []
    for name in configs:
        for dim in (12, 20):
            rows = [r for r in results if r['config']==name and r['feature_dim']==dim]
            summary.append(dict(config=name, feature_dim=dim,
                mean_macro_f1=float(np.mean([r['macro_f1'] for r in rows])),
                std_macro_f1=float(np.std([r['macro_f1'] for r in rows])),
                mean_drowsy_recall=float(np.mean([r['drowsy_recall'] for r in rows]))))
    (output / 'summary.json').write_text(json.dumps(summary, indent=2))
    (output / 'complete.json').write_text(json.dumps(dict(runs=len(results), test_evaluated=False)))
    print(json.dumps(summary, indent=2))


In [ ]:
from datetime import datetime, timezone
INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
candidates = [p.parent for p in INPUT.rglob('metadata.json') if p.parent.name == 'processed20_v4']
if len(candidates) != 1:
    raise ValueError('Attach exactly one processed20_v4 dataset')
DATA = candidates[0]
RUN = WORK / ('validation_experiment_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
run_experiments(DATA, RUN)
print('Download this output directory:', RUN)
